# NB10 — Cross-Dataset Transfer (leakage-controlled)

Trains the proposed config (full-FT + FGM, the NB08/09 winner) on each binary corpus and evaluates on every
other corpus's **test** split. **Cross-corpus leakage control (Requirement 2):** before any off-diagonal
"train-on-A / eval-on-B" cell, every B-test row whose normalized text appears anywhere in corpus A is
removed, and the removed count is logged. Diagonal (in-domain) cells use B-test as-is (already internally
leak-free from NB01).

Binary corpora only (shared 0/1 label space): `ben_sarc_binary`, `banglasarc_binary`, `banglasarc3_binary`.
GPU notebook. Resumable.


In [1]:
import importlib, subprocess, sys
def ensure(p,i=None):
    try: importlib.import_module(i or p)
    except ImportError: subprocess.run([sys.executable,"-m","pip","install","-q",p],check=False)
for pkg,imp in [("transformers","transformers"),("accelerate","accelerate"),("scikit-learn","sklearn"),
                ("pandas","pandas"),("numpy","numpy")]: ensure(pkg,imp)
print("deps ok")

deps ok


In [2]:
import os, json, time, random, shutil, warnings, unicodedata, re, gc
from pathlib import Path
import numpy as np, pandas as pd, torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings("ignore")
HAS_CUDA=torch.cuda.is_available(); HAS_BF16=HAS_CUDA and torch.cuda.is_bf16_supported()
print("cuda:",HAS_CUDA,"| bf16:",HAS_BF16)
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if HAS_CUDA: torch.cuda.manual_seed_all(s)
SEED=42; set_seed(SEED)

def find_repo_root():
    for c in [Path("/workspace/Sarcasm_detection"), Path.cwd()] + list(Path.cwd().resolve().parents):
        if (c/"01_data").exists(): return c.resolve()
    raise RuntimeError("repo root not found.")
ROOT=find_repo_root(); SPLITS=ROOT/"01_data"/"interim"/"splits"
CKPT=ROOT/"03_checkpoints"; TABLES=ROOT/"04_outputs"/"tables"
for p in [CKPT,TABLES]: p.mkdir(parents=True,exist_ok=True)
assert SPLITS.exists(), "Run NB01 first."

_ZW=dict.fromkeys(map(ord,["\u200b","\u200c","\u200d","\ufeff"]),None)
def norm_key(s):
    if not isinstance(s,str): s="" if pd.isna(s) else str(s)
    s=unicodedata.normalize("NFC",s).translate(_ZW); return re.sub(r"\s+"," ",s).strip().casefold()
class DS(torch.utils.data.Dataset):
    def __init__(self,texts,labels,tok,ml=128):
        self.enc=tok(list(texts),truncation=True,padding="max_length",max_length=ml,return_tensors="pt")
        self.labels=torch.tensor(list(labels),dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self,i):
        it={k:v[i] for k,v in self.enc.items()}; it["labels"]=self.labels[i]; return it
def load_corpus(task):
    def rd(sp):
        df=pd.read_csv(SPLITS/f"{task}_{sp}.csv")[["text","label_binary"]].dropna(subset=["text"]).copy()
        df["text"]=df["text"].astype(str); df["label_binary"]=df["label_binary"].astype(int)
        df["nk"]=df["text"].map(norm_key); return df
    return rd("train"),rd("val"),rd("test")

cuda: True | bf16: True


In [3]:
# FGM + adversarial trainer (proposed config)
class FGM:
    def __init__(self,model,eps=0.5,emb="word_embeddings"):
        self.m=model; self.eps=eps; self.emb=emb; self.bk={}
    def attack(self):
        for n,p in self.m.named_parameters():
            if p.requires_grad and self.emb in n and p.grad is not None:
                self.bk[n]=p.data.clone(); nm=torch.norm(p.grad)
                if nm!=0 and not torch.isnan(nm): p.data.add_(self.eps*p.grad/nm)
    def restore(self):
        for n,p in self.m.named_parameters():
            if n in self.bk: p.data=self.bk[n]
        self.bk={}
class AdvTrainer(Trainer):
    def __init__(self,*a,fgm_eps=0.5,**k):
        super().__init__(*a,**k); self._fgm=FGM(self.model,fgm_eps)
    def training_step(self,model,inputs,num_items_in_batch=None):
        model.train(); inputs=self._prepare_inputs(inputs)
        with self.compute_loss_context_manager(): loss=self.compute_loss(model,inputs)
        if self.args.n_gpu>1: loss=loss.mean()
        self.accelerator.backward(loss)
        self._fgm.attack()
        with self.compute_loss_context_manager(): la=self.compute_loss(model,inputs)
        if self.args.n_gpu>1: la=la.mean()
        self.accelerator.backward(la); self._fgm.restore()
        return loss.detach()
def cm(ep):
    logits,labels=ep; preds=np.asarray(logits).argmax(-1)
    return {"macro_f1":f1_score(labels,preds,average="macro",zero_division=0)}
def build_args(out,epochs=8,batch=32,lr=2e-5):
    bf16=HAS_BF16; fp16=(HAS_CUDA and not HAS_BF16)  # bf16 keeps adv stable; fp32 fallback if neither
    if not HAS_BF16 and HAS_CUDA: fp16=False  # force fp32 for adversarial stability on non-Ampere
    common=dict(output_dir=str(out),num_train_epochs=epochs,per_device_train_batch_size=batch,
                per_device_eval_batch_size=batch*2,learning_rate=lr,weight_decay=0.01,warmup_ratio=0.1,
                save_strategy="epoch",logging_strategy="epoch",save_total_limit=1,load_best_model_at_end=True,
                metric_for_best_model="macro_f1",greater_is_better=True,report_to="none",seed=SEED,
                data_seed=SEED,fp16=fp16,bf16=bf16)
    try: return TrainingArguments(evaluation_strategy="epoch",**common)
    except TypeError: return TrainingArguments(eval_strategy="epoch",**common)

# proposed FGM epsilon (from NB09 if available)
fc=TABLES/"09_final_config.json"; FGM_EPS=0.5
if fc.exists():
    j=json.load(open(fc)); FGM_EPS=j.get("fgm_eps") or 0.5
print("proposed config: full-FT + FGM, eps =", FGM_EPS)
MODEL="csebuetnlp/banglabert"; tok=AutoTokenizer.from_pretrained(MODEL)
CORPora=["ben_sarc_binary","banglasarc_binary","banglasarc3_binary"]

proposed config: full-FT + FGM, eps = 0.5


In [4]:
def train_on(source):
    set_seed(SEED); tr,va,te=load_corpus(source)
    tr_ds=DS(tr["text"],tr["label_binary"],tok); va_ds=DS(va["text"],va["label_binary"],tok)
    out=CKPT/f"10_src_{source}"
    if out.exists(): shutil.rmtree(out,ignore_errors=True)
    model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=2)
    trn=AdvTrainer(model=model,args=build_args(out),train_dataset=tr_ds,eval_dataset=va_ds,
                   compute_metrics=cm,callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],fgm_eps=FGM_EPS)
    trn.train()
    return trn, {"train":tr,"val":va,"test":te}

def eval_cross(trn, source, source_dfs, target):
    _,_,te_t=load_corpus(target)
    n_total=len(te_t); n_removed=0
    if target!=source:
        a_keys=set(source_dfs["train"]["nk"])|set(source_dfs["val"]["nk"])|set(source_dfs["test"]["nk"])
        keep=~te_t["nk"].isin(a_keys); n_removed=int((~keep).sum()); te_t=te_t[keep]
    if len(te_t)==0: return None
    ds=DS(te_t["text"],te_t["label_binary"],tok)
    logits=trn.predict(ds).predictions; preds=np.asarray(logits).argmax(-1)
    y=te_t["label_binary"].values
    return {"source":source,"target":target,"in_domain":source==target,
            "n_eval_total":n_total,"n_removed_leak":n_removed,"n_eval_clean":len(te_t),
            "macro_f1":float(f1_score(y,preds,average="macro",zero_division=0)),
            "accuracy":float(accuracy_score(y,preds))}

In [5]:
OUT=TABLES/"10_cross_dataset_matrix.csv"
rows=[]; done=set()
if OUT.exists():
    prev=pd.read_csv(OUT); rows=prev.to_dict("records"); done={(r["source"],r["target"]) for r in rows}; print("resuming:",len(done))
for source in CORPora:
    if all((source,t) in done for t in CORPora): print("skip source (done):",source); continue
    print("\n"+"="*70+f"\nTRAIN on {source}\n"+"="*70)
    trn, sdfs = train_on(source)
    for target in CORPora:
        if (source,target) in done: print("  skip",source,"->",target); continue
        r=eval_cross(trn, source, sdfs, target)
        if r: rows.append(r); pd.DataFrame(rows).to_csv(OUT,index=False)
        print(f"  {source} -> {target}: macroF1={r['macro_f1']:.4f} (removed {r['n_removed_leak']} leaked rows)")
    del trn; gc.collect()
    if HAS_CUDA: torch.cuda.empty_cache()
    shutil.rmtree(CKPT/f"10_src_{source}",ignore_errors=True)
print("\nDONE ->", OUT.name)


TRAIN on ben_sarc_binary


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1
1,0.558400,0.449424,0.795023
2,0.405600,0.441024,0.804367
3,0.264100,0.494523,0.788111
4,0.141100,0.567917,0.798031


  ben_sarc_binary -> ben_sarc_binary: macroF1=0.7992 (removed 0 leaked rows)


  ben_sarc_binary -> banglasarc_binary: macroF1=0.5908 (removed 0 leaked rows)


...

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1
1,0.628800,0.524397,0.754079
2,0.482700,0.503558,0.778442
3,0.344500,0.537435,0.763407
4,0.214500,0.584837,0.755741


  banglasarc3_binary -> ben_sarc_binary: macroF1=0.6780 (removed 0 leaked rows)


  banglasarc3_binary -> banglasarc_binary: macroF1=0.6052 (removed 28 leaked rows)


  banglasarc3_binary -> banglasarc3_binary: macroF1=0.7659 (removed 0 leaked rows)

DONE -> 10_cross_dataset_matrix.csv


In [6]:
df=pd.read_csv(OUT)
piv=df.pivot(index="source",columns="target",values="macro_f1")
piv.to_csv(TABLES/"10_cross_dataset_pivot.csv")
print("="*70); print("  CROSS-DATASET MACRO-F1 (rows=train, cols=eval; diagonal=in-domain)"); print("="*70)
print(piv.to_string(float_format="%.4f"))
print("\nLeakage rows removed per off-diagonal cell:")
print(df[df["in_domain"]==False][["source","target","n_removed_leak","n_eval_clean"]].to_string(index=False))
diag=df[df["in_domain"]].macro_f1.mean(); off=df[~df["in_domain"]].macro_f1.mean()
print(f"\nmean in-domain={diag:.4f}  |  mean cross-domain={off:.4f}  |  transfer drop={diag-off:.4f}")

  CROSS-DATASET MACRO-F1 (rows=train, cols=eval; diagonal=in-domain)
target              banglasarc3_binary  banglasarc_binary  ben_sarc_binary
source                                                                    
banglasarc3_binary              0.7659             0.6052           0.6780
banglasarc_binary               0.3384             0.9766           0.3464
ben_sarc_binary                 0.6407             0.5908           0.7992

Leakage rows removed per off-diagonal cell:
            source             target  n_removed_leak  n_eval_clean
   ben_sarc_binary  banglasarc_binary               0           464
   ben_sarc_binary banglasarc3_binary               0           791
 banglasarc_binary    ben_sarc_binary               0          2563
 banglasarc_binary banglasarc3_binary              29           762
banglasarc3_binary    ben_sarc_binary               0          2563
banglasarc3_binary  banglasarc_binary              28           436

mean in-domain=0.8472  |  mean cro